In [1]:
import awkward as ak


In [2]:
data = ak.from_parquet("/scratch/persistent/laurits/ml-tau/20260818_tauDaughterDataset/z_train.parquet")

In [3]:
import awkward as ak

arr = ak.from_parquet(
    "/scratch/persistent/laurits/ml-tau/20260818_tauDaughterDataset/z_train.parquet",
    row_groups=list(range(10)),
)

In [60]:
mask = ak.sum(abs(arr.gen_jet_tau_vis_daughter_pdgs) == 11, axis = 1) == 0

In [62]:
arr = arr[mask]

In [82]:
import awkward as ak

a = arr.gen_jet_tau_vis_daughter_pdgs

def get_decay_mode_id(daughter_pdgs):
    # keys = np.unique(ak.flatten(abs(arr.gen_jet_tau_vis_daughter_pdgs)))
    keys = [22, 111, 130, 211, 221, 223, 310, 311, 321, 323]  # Should get the same result as above, but this is just a failsafe.
    
    targets = ak.Array([
        # 22 111 130 211 221 223 310 311 321 323
        [0,  1,  0,  1,  0,  0,  0,  0,  0,  0],  # 0: pi + pi0
        [0,  0,  0,  1,  0,  0,  0,  0,  0,  0],  # 1: pi
        [0,  0,  0,  3, 0,  0,  0,  0,  0,  0],  # 2: 3pi
        [0,  2,  0,  1, 0,  0,  0,  0,  0,  0],  # 3: pi + 2pi0
        [0,  1,  0,  3, 0,  0,  0,  0,  0,  0],  # 4: 3pi + pi0
        [0,  3,  0,  1, 0,  0, 0,   0,  0,  0],  # 5: pi + 3pi0
        [0,  0,  0,  1, 0,  0, 1,   0,  0,  0],  # 6: pi + K0
        [0,  0,  0,  0, 0,  0, 1,   0,  0,  0],  # 7: K
        [0,  2,  0,  3, 0,  0,  0,  0,  0,  0],  # 8: 3pi + 2pi0
        [0,  1, 0,   0, 0,  0, 1,   0,  0,  0],  # 9: K + pi0
        [0,  1, 0,   1, 0,  0, 1,   0,  0,  0],  # 10: pi + pi0 + K0
        [0,  0, 0,   2, 0,  0, 0,   0, 1,  0],  # 11: 2pi + K
    ])
    
    counts = ak.zip({
        f"n_{k}": ak.sum(abs(a) == k, axis=1)
        for k in keys
    })
    signature = ak.zeros_like(counts.n_22)
    for k in keys:
        signature = signature * 10 + counts[f"n_{k}"]
    
    # Encode the target configurations using the exact same scheme.
    target_signature = ak.zeros_like(targets[:, 0])
    
    for i in range(len(keys)):
        target_signature = target_signature * 10 + targets[:, i]
    
    # Match each jet against the 12 target signatures.
    matches = signature[:, None] == target_signature[None, :]
    
    # ID of matching target.
    class_id = ak.argmax(matches, axis=1, mask_identity=False)
    
    # No match -> Other = -1
    class_id = ak.where(
        ak.any(matches, axis=1),
        class_id,
        15,
    )
    return class_id

In [85]:
class_id = get_decay_mode_id(a)
new_data = ak.with_field(arr, class_id, "gen_jet_tau_decay_mode_rare")

In [86]:
new_data

<Array [{reco_cand_pdgs: [...], ...}, ...] type='10239 * {reco_cand_pdgs: v...'>

In [87]:
new_data.fields

['reco_cand_pdgs',
 'reco_cand_charges',
 'gen_jet_tau_vis_energy',
 'gen_jet_tau_decaymode',
 'gen_jet_tau_charge',
 'gen_jet_tau_DV_x',
 'gen_jet_tau_DV_y',
 'gen_jet_tau_DV_z',
 'gen_jet_tau_vis_daughter_p4s',
 'gen_jet_tau_vis_daughter_pdgs',
 'gen_jet_tau_vis_daughter_charges',
 'reco_cand_dxy',
 'reco_cand_dz',
 'reco_cand_d3',
 'reco_cand_d0',
 'reco_cand_z0',
 'reco_cand_dxy_2d',
 'reco_cand_dz_2d',
 'reco_cand_d3_2d',
 'reco_cand_pca_x',
 'reco_cand_pca_y',
 'reco_cand_pca_z',
 'reco_cand_vertex_x',
 'reco_cand_vertex_y',
 'reco_cand_vertex_z',
 'reco_cand_phi0',
 'reco_cand_tanL',
 'reco_cand_omega',
 'reco_cand_dxy_error',
 'reco_cand_dz_error',
 'reco_cand_d3_error',
 'reco_cand_d0_error',
 'reco_cand_z0_error',
 'reco_cand_dxy_2d_error',
 'reco_cand_dz_2d_error',
 'reco_cand_d3_2d_error',
 'reco_cand_pca_x_error',
 'reco_cand_pca_y_error',
 'reco_cand_pca_z_error',
 'reco_cand_signed_dxy',
 'reco_cand_signed_dz',
 'reco_cand_signed_d3',
 'reco_cand_signed_d0',
 'reco_cand_

In [88]:
import os
import glob
import awkward as ak


DETR_dataset_dir = "/scratch/persistent/laurits/ml-tau/20260818_tauDaughterDataset/"
rare_decays_dir = "/scratch/persistent/laurits/ml-tau/20260824_rareDecaysDataset/"

In [89]:
list(glob.glob(os.path.join(DETR_dataset_dir, "*.parquet")))

['/scratch/persistent/laurits/ml-tau/20260818_tauDaughterDataset/z_train.parquet',
 '/scratch/persistent/laurits/ml-tau/20260818_tauDaughterDataset/qq_train.parquet',
 '/scratch/persistent/laurits/ml-tau/20260818_tauDaughterDataset/qq_test.parquet',
 '/scratch/persistent/laurits/ml-tau/20260818_tauDaughterDataset/z_test.parquet']